In [ ]:
import os

# 1) Define your token, repo URL and local directory name
REPO_URL  = f"https://github.com/semilleroCV/BreastCATT.git"
REPO_DIR  = "BreastCATT"

# 2) Clone or pull depending on whether the folder exists
if not os.path.isdir(REPO_DIR):
    print(f"Cloning repository into ./{REPO_DIR}…")
    os.system(f"git clone {REPO_URL}")
else:
    print(f"Directory '{REPO_DIR}' already exists. Pulling latest changes…")
    # If your origin remote wasn’t set with the token, you could uncomment:
    # os.system(f"git -C {REPO_DIR} remote set-url origin {REPO_URL}")
    os.system(f"git -C {REPO_DIR} pull")

# 3) Change into the repo directory
os.chdir(REPO_DIR)
print("Current working directory:", os.getcwd())

Cloning repository into ./BreastCATT…


Cloning into 'BreastCATT'...


Current working directory: /kaggle/working/BreastCATT


In [2]:
!git fetch origin
!git pull origin main

From https://github.com/semilleroCV/BreastCATT
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
# every time you do a factory reset you have to run this
!pip install -r requirements.txt -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 85.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 66.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# log in hugging face hug to load data and models
from huggingface_hub import login

login(token="", add_to_git_credential=True)

Token has not been saved to git credential helper.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.


In [1]:
from datasets import load_dataset, DatasetDict
dataset = load_dataset("SemilleroCV/DMR-IR")
test = dataset['test']

/home/guillermo/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# IF YOU ARE WORKING ON KAGGLE OR GOOGLE COLAB DO NOT EXECUTE THIS CELL

import sys
import os

# El notebook está en 'notebooks/', pero el proyecto (y `breastcatt`) está en el directorio padre.
# Agregamos el directorio padre al path para poder importar `breastcatt`.
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
  sys.path.append(project_root)

In [3]:
import torch
from datasets import Dataset, Image, ClassLabel
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, Resize, ToTensor, Lambda
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score
from torchmetrics.classification import BinarySpecificity
from huggingface_hub import hf_hub_download
import json

from breastcatt import tfvit

In [4]:
# --- Transformations ---
min_max_norm = Lambda(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8) if x.max() > x.min() else x)
transforms = Compose([
    Resize((224, 224)),
    ToTensor(),
    min_max_norm,
])

In [5]:
def preprocess_val(example_batch):
    """Apply _val_transforms across a batch."""
    example_batch["pixel_values"] = [
        transforms(image) for image in example_batch["image"]
    ]
    return example_batch

full_test_dataset = test.with_transform(preprocess_val)

In [6]:
def collate_fn(examples):
        pixel_values = torch.stack([example["pixel_values"] for example in examples])
        texts = [example["text"] for example in examples]
        labels = torch.tensor([example["label"] for example in examples])
        return {"pixel_values": pixel_values, "texts": texts, "labels": labels} 

In [7]:
BATCH_SIZE = 8
test_dataloader = DataLoader(dataset = full_test_dataset, batch_size = BATCH_SIZE,
                             shuffle = False, collate_fn = collate_fn)

In [8]:
def evaluate_split(model, dataloader, device):
    """
    Evaluates the model on a given dataloader and computes metrics.
    """
    model.eval()
    all_preds = []
    all_labels = []
    specificity_metric = BinarySpecificity().to(device)

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            pixel_values = batch["pixel_values"].to(device)
            texts = batch["texts"]
            labels = batch["labels"].to(device)

            outputs = model(pixel_values=pixel_values, texts=texts, labels=labels)
            
            # Assuming binary classification with logits and BCEWithLogitsLoss
            # Convert logits to probabilities and then to predictions
            probs = torch.sigmoid(outputs.logits).squeeze()
            preds = (probs > 0.5).long()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            specificity_metric.update(preds, labels)

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    sensitivity = recall_score(all_labels, all_preds, zero_division=0) # Recall is Sensitivity
    specificity = specificity_metric.compute().item()

    return {
        "accuracy": accuracy,
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
    }

In [9]:
# --- Configuration ---
MODEL_REPO_ID = "SemilleroCV/tfvit-base_seg_text" # Change to your model repo ID
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
# --- Load Model ---
print(f"Loading model from {MODEL_REPO_ID}...")
config_path = hf_hub_download(repo_id=MODEL_REPO_ID, filename="config.json")
state_dict_path = hf_hub_download(repo_id=MODEL_REPO_ID, filename="pytorch_model.bin")

with open(config_path, "r") as f:
    init_args = json.load(f)

Loading model from SemilleroCV/tfvit-base_seg_text...


In [11]:
# The checkpoint on the hub was trained for 1 class (binary)
init_args['num_classes'] = 1 

model = tfvit.multimodal_vit_base_patch16(**init_args)
state_dict = torch.load(state_dict_path, map_location="cpu")
model.load_state_dict(state_dict)
model.to(DEVICE)
print("Model loaded successfully.")

✅ MAE base weights already exist at: checkpoints/fvit/mae_pretrain_vit_base.pth
✅ TransUNet weights already exist at: checkpoints/segmentation/lucky-sweep-6_0.4937.pth
Loading checkpoint from: checkpoints/fvit/mae_pretrain_vit_base.pth
Adapting patch_embed.proj.weight from 3 channels to 1 channel...

✅ Loaded weights: 148 layers.
❌ Not found in model: 0 layers.
📋 Details of load_state_dict:
_IncompatibleKeys(missing_keys=['cls_token', 'pos_embed', 'language_model.model_lm.embeddings.word_embeddings.weight', 'language_model.model_lm.embeddings.position_embeddings.weight', 'language_model.model_lm.embeddings.token_type_embeddings.weight', 'language_model.model_lm.encoder.layer.0.attention.ln.weight', 'language_model.model_lm.encoder.layer.0.attention.ln.bias', 'language_model.model_lm.encoder.layer.0.attention.self.query.weight', 'language_model.model_lm.encoder.layer.0.attention.self.query.bias', 'language_model.model_lm.encoder.layer.0.attention.self.key.weight', 'language_model.model_

In [12]:
metrics = evaluate_split(model, test_dataloader, DEVICE)

print(f"  Accuracy:    {metrics['accuracy']:.4f}")
print(f"  Precision:   {metrics['precision']:.4f}")
print(f"  Sensitivity: {metrics['sensitivity']:.4f}")
print(f"  Specificity: {metrics['specificity']:.4f}")

Evaluating:   0%|          | 0/184 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 184/184 [02:26<00:00,  1.26it/s]

  Accuracy:    0.6259
  Precision:   0.0000
  Sensitivity: 0.0000
  Specificity: 1.0000
